In [5]:
import pandas as pd
import ssl

In [6]:
ssl._create_default_https_context = ssl._create_unverified_context

url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$limit=150000&$order=date%20DESC"
df = pd.read_csv(url)

print(df.shape)
df.head()

(150000, 22)


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14227673,JK291624,2026-06-12T00:00:00.000,071XX S WASHTENAW AVE,0460,BATTERY,SIMPLE,APARTMENT,False,True,...,17,66.0,08B,1159607.0,1857244.0,2026,2026-06-19T15:43:57.000,41.763990,-87.690566,"\n, \n(41.763990085, -87.690565528)"
1,14230986,JK295947,2026-06-12T00:00:00.000,024XX W TAYLOR ST,0920,MOTOR VEHICLE THEFT,ATTEMPT - AUTOMOBILE,CHA PARKING LOT / GROUNDS,False,False,...,28,28.0,07,1160367.0,1895639.0,2026,2026-06-19T15:43:57.000,41.869335,-87.686721,"\n, \n(41.869335009, -87.686720968)"
2,14230786,JK295690,2026-06-12T00:00:00.000,008XX S PLYMOUTH CT,0820,THEFT,$500 AND UNDER,APARTMENT,False,False,...,4,32.0,06,1176182.0,1896753.0,2026,2026-06-19T15:43:57.000,41.872050,-87.628627,"\n, \n(41.872050318, -87.628626791)"
3,14230143,JK294801,2026-06-12T00:00:00.000,048XX W MADISON ST,0820,THEFT,$500 AND UNDER,STREET,False,False,...,28,25.0,06,1144224.0,1899588.0,2026,2026-06-19T15:43:57.000,41.880490,-87.745887,"\n, \n(41.880489945, -87.745887499)"
4,14232164,JK297299,2026-06-12T00:00:00.000,054XX S CORNELL AVE,0820,THEFT,$500 AND UNDER,RESIDENCE,False,False,...,5,41.0,06,1188172.0,1869776.0,2026,2026-06-19T15:43:57.000,41.797745,-87.585470,"\n, \n(41.797745447, -87.585469563)"


In [7]:
import plotly.express as px
import pandas as pd

# ── 1. Limpieza básica ───────────────────────────────────────────
df_mapa = df.dropna(subset=['latitude', 'longitude', 'primary_type']).copy()
df_mapa['latitude'] = pd.to_numeric(df_mapa['latitude'], errors='coerce')
df_mapa['longitude'] = pd.to_numeric(df_mapa['longitude'], errors='coerce')
df_mapa = df_mapa.dropna(subset=['latitude', 'longitude'])

# ── 2. Agregación por zona y tipo de crimen ──────────────────────
df_agg = (
    df_mapa.groupby(['community_area', 'primary_type'], as_index=False)
    .agg(
        total_crimenes=('primary_type', 'count'),
        lat=('latitude', 'mean'),
        lon=('longitude', 'mean')
    )
)

tipos = sorted(df_agg['primary_type'].unique())

# ── 3. Mapa con dropdown ─────────────────────────────────────────
import plotly.graph_objects as go

fig = go.Figure()

for tipo in tipos:
    subset = df_agg[df_agg['primary_type'] == tipo]
    fig.add_trace(go.Scattermapbox(
        lat=subset['lat'],
        lon=subset['lon'],
        mode='markers',
        marker=dict(
            size=subset['total_crimenes'] / subset['total_crimenes'].max() * 30 + 5,
            color=subset['total_crimenes'],
            colorscale='Reds',
            showscale=True,
            colorbar=dict(title='Crímenes')
        ),
        text=subset['community_area'].astype(str) + '<br>Crímenes: ' + subset['total_crimenes'].astype(str),
        hoverinfo='text',
        name=tipo,
        visible=(tipo == tipos[0])  # Solo el primero visible al inicio
    ))

# Botones del dropdown
buttons = []
for i, tipo in enumerate(tipos):
    visibility = [j == i for j in range(len(tipos))]
    buttons.append(dict(
        label=tipo,
        method='update',
        args=[{'visible': visibility},
              {'title': f'Crímenes en Chicago — {tipo}'}]
    ))

fig.update_layout(
    title=f'Crímenes en Chicago — {tipos[0]}',
    mapbox=dict(
        style='carto-positron',
        center=dict(lat=41.8781, lon=-87.6298),
        zoom=9.5
    ),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.01,
        y=0.99,
        xanchor='left',
        yanchor='top'
    )],
    margin=dict(r=0, t=50, l=0, b=0),
    height=650
)

fig.show()
fig.write_html("mapa_crimenes_chicago.html")

/var/folders/zr/gtbmvgdn5dj_b1tblkbm659h0000gn/T/ipykernel_17006/273880658.py:29: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/zr/gtbmvgdn5dj_b1tblkbm659h0000gn/T/ipykernel_17006/273880658.py:29: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/zr/gtbmvgdn5dj_b1tblkbm659h0000gn/T/ipykernel_17006/273880658.py:29: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/zr/gtbmvgdn5dj_b1tblkbm659h0000gn/T/ipykernel_17006/273880658.py:29: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go

In [15]:
df_agg.sort_values('total_crimenes', ascending=False)

,community_area,primary_type,total_crimenes,lat,lon
184,8.0,THEFT,2851,41.897000,-87.631179
734,32.0,THEFT,1988,41.880178,-87.628151
635,28.0,THEFT,1877,41.876997,-87.657588
540,25.0,BATTERY,1689,41.889846,-87.759267
135,6.0,THEFT,1572,41.942880,-87.653526
...,...,...,...,...,...
1575,71.0,CONCEALED CARRY LICENSE VIOLATION,1,41.746346,-87.653628
197,9.0,PUBLIC PEACE VIOLATION,1,42.017872,-87.812668
196,9.0,PROSTITUTION,1,42.011703,-87.807159
1159,52.0,KIDNAPPING,1,41.696299,-87.531660
